# Public API N-Glycosylation Walkthrough

This notebook builds a 5FYJ N-glycosylated protein through the stable public conjugation API:

```python
from polyzymd.builders.conjugation import ConjugateBuildRequest, build_conjugate
```

The executable default uses one real 5FYJ target site, `B:ASN625`, and a GlcNAc/NAG SMILES moiety. The POC directory currently includes one larger glycan SDF fixture; this notebook shows how to convert that SDF to SMILES for provenance, but passes an embedding-friendly NAG SMILES to PolyzyMD so the public request path remains reproducible on a CPU build environment.

The final handoff/export structure is exposed as `result.solvated_pdb_path`. The minimized and short-equilibrated conjugate smoke artifacts are exposed separately as `result.minimized_conjugate_pdb_path`, `result.equilibrated_conjugate_pdb_path`, and `result.relaxed_conjugate_pdb_path`.

In [1]:
import importlib
import sys
from pathlib import Path

from rdkit import Chem


def find_repo_root() -> Path:
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "pyproject.toml").exists() and (base / "src/polyzymd").is_dir():
            return base
    raise FileNotFoundError("Could not locate the PolyzyMD repository root")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

conjugation_api = importlib.import_module("polyzymd.builders.conjugation")
ConjugateBuildRequest = conjugation_api.ConjugateBuildRequest
build_conjugate = conjugation_api.build_conjugate

POC_DIR = REPO_ROOT / "src/polyzymd/builders/conjugation/poc"
RAW_5FYJ_PDB = POC_DIR / "5fyj-monomer.pdb"
GLYCAN_SDF = POC_DIR / "data/test_glycan.sdf"
OUTPUT_DIR = POC_DIR / "output/public-api-n-glycosylation"
PREPARED_PROTEIN_PDB = OUTPUT_DIR / "5fyj-monomer-protein-only.pdb"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"POC directory: {POC_DIR}")
print(f"Raw 5FYJ monomer: {RAW_5FYJ_PDB}")
print(f"Glycan SDF fixture: {GLYCAN_SDF}")
print(f"Output directory: {OUTPUT_DIR}")

Repository root: /home/joelaforet/Shirts-Lab-Linux/polyzymd
POC directory: /home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc
Raw 5FYJ monomer: /home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/5fyj-monomer.pdb
Glycan SDF fixture: /home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/data/test_glycan.sdf
Output directory: /home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation


## Prepare The 5FYJ Protein Input

The bundled 5FYJ monomer fixture includes crystallographic waters, citrate, ethylene glycol, and historical glycans. The direct public request expects a cleaned protein PDB; it canonicalizes hydrogens before resolving the Asn ND2 leaving hydrogen. This notebook writes a reproducible protein-only PDB containing standard amino-acid residues and preserving the original 5FYJ chain IDs.

Target-site chain IDs in this fixture are `B:ASN625`, `G:ASN234`, and `G:ASN332`. These correspond to Hannah's glycan labels F, P, and X on the historical glycosylated PDB.

In [2]:
STANDARD_AMINO_ACIDS = {
    "ALA",
    "ARG",
    "ASN",
    "ASP",
    "CYS",
    "GLN",
    "GLU",
    "GLY",
    "HIS",
    "ILE",
    "LEU",
    "LYS",
    "MET",
    "PHE",
    "PRO",
    "SER",
    "THR",
    "TRP",
    "TYR",
    "VAL",
}


def pdb_summary(path: Path) -> dict[str, object]:
    atom_count = 0
    residues = set()
    chains = set()
    residue_names = set()
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.startswith(("ATOM", "HETATM")):
            continue
        atom_count += 1
        chain_id = line[21].strip() or "?"
        residue_name = line[17:20].strip()
        residue_number = line[22:26].strip()
        insertion_code = line[26].strip()
        chains.add(chain_id)
        residue_names.add(residue_name)
        residues.add((chain_id, residue_number, insertion_code, residue_name))
    return {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "atom_count": atom_count,
        "residue_count": len(residues),
        "chains": sorted(chains),
        "contains_ASX": "ASX" in residue_names,
        "contains_NAG": "NAG" in residue_names,
        "contains_water": bool({"HOH", "WAT", "SOL"} & residue_names),
    }


def write_standard_protein_only_pdb(source: Path, destination: Path) -> Path:
    lines = []
    for line in source.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.startswith(("ATOM", "HETATM")):
            continue
        if line[17:20].strip() in STANDARD_AMINO_ACIDS:
            lines.append(line + "\n")
    lines.append("END\n")
    destination.write_text("".join(lines), encoding="utf-8")
    return destination


write_standard_protein_only_pdb(RAW_5FYJ_PDB, PREPARED_PROTEIN_PDB)
{"raw": pdb_summary(RAW_5FYJ_PDB), "prepared": pdb_summary(PREPARED_PROTEIN_PDB)}

Out[0]: 
{'raw': {'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/5fyj-monomer.pdb',
  'size_bytes': 401474,
  'atom_count': 5081,
  'residue_count': 647,
  'chains': ['B', 'G'],
  'contains_ASX': False,
  'contains_NAG': True,
  'contains_water': True},
 'prepared': {'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation/5fyj-monomer-protein-only.pdb',
  'size_bytes': 394135,
  'atom_count': 4989,
  'residue_count': 633,
  'chains': ['B', 'G'],
  'contains_ASX': False,
  'contains_NAG': False,
  'contains_water': False}}


## Glycan SMILES Inputs

PolyzyMD core accepts SMILES moieties for this public request path. SDF-to-SMILES conversion is kept in the tutorial layer with RDKit. The larger `test_glycan.sdf` fixture is modest and useful for provenance, but the executable default uses a fixed GlcNAc/NAG SMILES because that single-residue moiety embeds reliably and keeps this notebook fast enough for routine validation.

In [3]:
sdf_supplier = Chem.SDMolSupplier(str(GLYCAN_SDF), removeHs=False)
sdf_mol = sdf_supplier[0]
assert sdf_mol is not None, f"Could not load {GLYCAN_SDF}"
SDF_DERIVED_GLYCAN_SMILES = Chem.MolToSmiles(Chem.RemoveHs(sdf_mol), isomericSmiles=True)

# Embedding-friendly GlcNAc/NAG monosaccharide used for the executable default.
NAG_SMILES = "CC(=O)N[C@@H]1[C@@H](O)[C@H](O)[C@@H](CO)O[C@H]1O"

{
    "sdf_fixture_atoms": sdf_mol.GetNumAtoms(),
    "sdf_derived_glycan_smiles": SDF_DERIVED_GLYCAN_SMILES,
    "executable_nag_smiles": NAG_SMILES,
}

Out[0]: 
{'sdf_fixture_atoms': 94,
 'sdf_derived_glycan_smiles': 'CC(=O)N[C@H]1[C@H](O[C@H]2[C@H](O)[C@@H](NC(C)=O)CO[C@@H]2CO)O[C@H](CO)[C@@H](O[C@@H]2O[C@H](CO[C@H]3O[C@H](CO[C@H]4O[C@H](CO)[C@@H](O)[C@H](O)[C@@H]4O)[C@@H](O)[C@H](O[C@H]4O[C@H](CO)[C@@H](O)[C@H](O)[C@@H]4O)[C@@H]3O)[C@@H](O)[C@H](O[C@H]3O[C@H](CO)[C@@H](O)[C@H](O)[C@@H]3O[C@H]3O[C@H](CO)[C@@H](O)[C@H](O)[C@@H]3O)[C@@H]2O)[C@@H]1O',
 'executable_nag_smiles': 'CC(=O)N[C@@H]1[C@@H](O)[C@H](O)[C@@H](CO)O[C@H]1O'}


## Build Through The Public API

The default run attaches one NAG moiety to `B:ASN625`, the real 5FYJ site associated with historical glycan chain F. The optional three-site block below is intentionally disabled by default because this repository currently carries one executable NAG SMILES fixture rather than distinct F/P/X glycan SMILES fixtures. Set `RUN_THREE_SITE = True` only when you want a slower smoke run that attaches the same NAG moiety to all three target Asn sites.

In [4]:
RUN_THREE_SITE = False

target_sites = [
    {
        "label": "chainF-glycan-asn625",
        "site": {"chain_id": "B", "residue_name": "ASN", "residue_number": 625},
    }
]
if RUN_THREE_SITE:
    target_sites.extend(
        [
            {
                "label": "chainP-glycan-asn234",
                "site": {"chain_id": "G", "residue_name": "ASN", "residue_number": 234},
            },
            {
                "label": "chainX-glycan-asn332",
                "site": {"chain_id": "G", "residue_name": "ASN", "residue_number": 332},
            },
        ]
    )

attachments = []
for target in target_sites:
    attachments.append(
        {
            "name": target["label"],
            "site": target["site"],
            "moiety": {
                "name": "glcnac-nag",
                "smiles": NAG_SMILES,
                "residue_name": "NAG",
            },
            "mechanism": {"name": "n_glycosylation"},
        }
    )

request = ConjugateBuildRequest(
    protein_pdb_path=PREPARED_PROTEIN_PDB,
    attachments=tuple(attachments),
    output_dir=OUTPUT_DIR,
    free_polymer_seed=2026,
)

result = build_conjugate(request)
result.model_dump(mode="json")

Out[0]: 
{'status': 'completed',
 'output_dir': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation'),
 'config_path': None,
 'crosslinked_conjugate_pdb_path': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation/conjugate-construction/assembled_crosslinked.pdb'),
 'minimized_conjugate_pdb_path': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation/conjugate-construction/assembled_minimized.pdb'),
 'equilibrated_conjugate_pdb_path': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation/conjugate-construction/assembled_equilibrated.pdb'),
 'relaxed_conjugate_pdb_path': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation/conjuga

/home/joelaforet/Shirts-Lab-Linux/polyzymd/.pixi/envs/build/lib/python3.12/site-packages/openff/pablo/_pdb.py:234: UserWarning: Input PDB has an atom ordering that cannot be represented in an OpenFF Topology. The atoms in this topology will not be in same order as those in PDB file
  warnings.warn(
/home/joelaforet/Shirts-Lab-Linux/polyzymd/.pixi/envs/build/lib/python3.12/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


## Validate Public Artifacts

These checks intentionally use only the public `ConjugationResult` paths and simple PDB text parsing. They verify that the product-state Asn residue (`ASX`), glycan residue (`NAG`), and solvated handoff structure were produced.

In [5]:
def require_nonempty_path(label: str, path: Path | None) -> Path:
    assert path is not None, f"{label} path was not reported"
    path = Path(path)
    assert path.exists(), f"{label} does not exist: {path}"
    assert path.stat().st_size > 0, f"{label} is empty: {path}"
    return path


validated_paths = {
    "crosslinked_conjugate_pdb": require_nonempty_path(
        "crosslinked conjugate PDB", result.crosslinked_conjugate_pdb_path
    ),
    "minimized_conjugate_pdb": require_nonempty_path(
        "minimized conjugate PDB", result.minimized_conjugate_pdb_path
    ),
    "relaxed_conjugate_pdb": require_nonempty_path(
        "relaxed conjugate PDB", result.relaxed_conjugate_pdb_path
    ),
    "handoff_solvated_pdb": require_nonempty_path(
        "handoff/export solvated PDB", result.solvated_pdb_path
    ),
    "workflow_json": require_nonempty_path(
        "workflow JSON sidecar", result.workflow_json_path
    ),
}

artifact_summaries = {
    name: pdb_summary(path) if path.suffix == ".pdb" else str(path)
    for name, path in validated_paths.items()
}

assert artifact_summaries["crosslinked_conjugate_pdb"]["contains_ASX"]
assert artifact_summaries["crosslinked_conjugate_pdb"]["contains_NAG"]
assert artifact_summaries["handoff_solvated_pdb"]["contains_ASX"]
assert artifact_summaries["handoff_solvated_pdb"]["contains_NAG"]
assert artifact_summaries["handoff_solvated_pdb"]["contains_water"]

artifact_summaries

Out[0]: 
{'crosslinked_conjugate_pdb': {'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation/conjugate-construction/assembled_crosslinked.pdb',
  'size_bytes': 807296,
  'atom_count': 9958,
  'residue_count': 634,
  'chains': ['A', 'C'],
  'contains_ASX': True,
  'contains_NAG': True,
  'contains_water': False},
 'minimized_conjugate_pdb': {'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation/conjugate-construction/assembled_minimized.pdb',
  'size_bytes': 807676,
  'atom_count': 9958,
  'residue_count': 634,
  'chains': ['A', 'B', 'C'],
  'contains_ASX': True,
  'contains_NAG': True,
  'contains_water': False},
 'relaxed_conjugate_pdb': {'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-n-glycosylation/conjugate-construction/assembled_equilibrated.pdb',
  'size_bytes': 807676,
  'atom_cou

## Handoff

Use `result.solvated_pdb_path` as the exported handoff PDB for later setup, simulation, or analysis workflows. Use `result.relaxed_conjugate_pdb_path` when you need the minimized/relaxed conjugate before solvation.